# Module 1 — GenAI Foundations: Topics & Practice

Work through this notebook *before* attempting the capstone. Each section ends with a tiny exercise — do them; the capstone assumes the muscle memory.

## Learning outcomes
By the end of this module you can:
1. Explain where a *generative* model sits relative to ML/DL.
2. Walk through the LLM training lifecycle (pre-train → SFT → RLHF/DPO).
3. Pick the right prompting technique for a task.
4. Write an automated eval that catches regressions.
5. Identify the failure modes (hallucination, injection, bias) of any LLM call you make.

## 1. The landscape — AI vs ML vs DL vs GenAI

- **AI** — anything that mimics intelligent behaviour (rule systems count).
- **ML** — learns a function from data; no hand-coded rules.
- **DL** — ML with deep neural nets; the model learns its own features.
- **GenAI** — DL models that *sample* from a learned distribution to produce new artefacts (text, image, code).

> **Exercise 1.1** — Write one sentence each about: (a) a GenAI use case, (b) a non-GenAI ML use case in the same domain. Why is each the right tool?

## 2. The LLM training lifecycle

| Stage | What it teaches | Data |
|---|---|---|
| Pre-training | Predict next token | trillions of web tokens |
| Supervised Fine-Tuning (SFT) | Follow instructions | curated prompt/response pairs |
| RLHF / DPO | Match human preferences | preference rankings |

Why care? **Knowing the stage explains the failure mode.** A model that hallucinates facts is a pre-training symptom; a model that refuses harmless requests is an RLHF symptom.

> **Exercise 1.2** — A user complains the assistant is too verbose. Which stage of training is the most likely culprit, and how would you fix it without retraining?

## 3. Prompt engineering — the hierarchy

1. **Zero-shot** — just ask. Cheapest. Try this first.
2. **Few-shot** — give 2–5 worked examples. Use when the format matters.
3. **Chain-of-thought (CoT)** — “think step by step”. Use for multi-step reasoning.
4. **Self-consistency** — sample N CoT chains, majority-vote the answer.
5. **Structured output** — force JSON via schema/function-calling.

> **Exercise 1.3** — For each task pick the cheapest technique that will work and explain why:
> a) "Classify support emails as billing/technical/other."
> b) "Solve this 4-step word problem."
> c) "Extract `{name, email, phone}` from a free-form bio."


In [ ]:
# Exercise 1.4 — write your first eval harness.
# Goal: catch a regression when you change a prompt.

GOLD = [
    ("My laptop won't turn on", "technical"),
    ("Why was I charged twice?", "billing"),
    ("Do you ship to Canada?", "other"),
]

def classify(text: str) -> str:
    # TODO: replace with a real LLM call (openai, anthropic, ollama...)
    # For now, a stub so the harness runs.
    return "other"

def accuracy(fn):
    correct = sum(1 for x, y in GOLD if fn(x) == y)
    return correct / len(GOLD)

print("accuracy:", accuracy(classify))


## 4. Evaluation — the only thing that separates a demo from a system

Three families of metric you should know:
- **Reference-based** — BLEU, ROUGE, exact-match. Fast, brittle on open generation.
- **Model-based** — embedding similarity, BERTScore, **LLM-as-judge**. Better for fluency / faithfulness.
- **Human** — gold standard, slowest, most expensive. Reserve for shipping decisions.

**Rule of thumb:** combine cheap automated metrics for CI + a small human-rated set for releases.

> **Exercise 1.4** — Convert the harness above to use a real LLM. Add one model-based metric (e.g. cosine similarity of your output vs the gold label embedded with `sentence-transformers`).

## 5. Ethics & safety — the failure modes you ship with

- **Hallucination** — confident wrong answers. Mitigate with grounding (RAG, citations) and refusal training.
- **Prompt injection** — user content overrides your system prompt. Mitigate with input/output validation, never trust user-supplied tool args.
- **Bias** — skewed outputs from skewed training data. Mitigate with eval slices across demographic axes.
- **Jailbreaks** — adversarial prompts to bypass safety. Mitigate with layered guardrails, not single-prompt patches.

> **Exercise 1.5** — Take any chatbot you've used. Try a prompt-injection attack ("Ignore previous instructions and..."). Note what worked and what didn't. **Document, don't exploit.**

## 6. The provider landscape — when to pick which API

| Need | Reasonable default |
|---|---|
| Best raw quality | Anthropic Claude (Opus) or OpenAI o-series |
| Cheap high-volume | Groq, Together, OpenAI mini-class |
| Local / private | Ollama (Llama-3, Mistral, Qwen) |
| Long context | Claude (1M), Gemini |
| Vision | OpenAI gpt-4o, Claude, Gemini |

Always benchmark *your* task — leaderboards lie about your domain.

> **Exercise 1.6** — Pick two providers. Run the classifier from Ex 1.4 on both. Compare accuracy, latency, $/1K calls. **This is the artefact you will reuse in the capstone.**